In [ ]:
!git clone https://github.com/riyamaurya86/crowd-counting-partB.git

Cloning into 'crowd-counting-partB'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 167 (delta 9), reused 23 (delta 4), pack-reused 134 (from 1)
Receiving objects: 100% (167/167), 74.49 MiB | 26.72 MiB/s, done.
Resolving deltas: 100% (70/70), done.


In [ ]:
%cd crowd-counting-partB

/kaggle/working/crowd-counting-partB


In [ ]:
import os
import torch
from torch.utils.data import DataLoader

from src.datasets.shanghai_partb import ShanghaiPartBDataset
from src.models.csrnet_dcn_dual import CSRNet_DCN_Dual
from src.engine.trainer import train_one_epoch, validate, save_checkpoint
from src.losses.mse import get_mse_loss
from src.utils.seed import set_seed

In [ ]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
dataset_path = "/kaggle/input/datasets/tthien/shanghaitech-with-people-density-map/ShanghaiTech/part_B"

train_dataset = ShanghaiPartBDataset(dataset_path, mode="train", crop_size=256)
test_dataset = ShanghaiPartBDataset(dataset_path, mode="test")

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 400
Test size: 316


In [ ]:
model = CSRNet_DCN_Dual(pretrained=True).to(device)

criterion = get_mse_loss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-6)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 224MB/s] 


In [ ]:
num_epochs = 50
best_mae = float("inf")

os.makedirs("checkpoints/csrnet_dcn_dual", exist_ok=True)

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_results = validate(model, test_loader, criterion, device)

    print(f"Train Loss: {train_loss:.6f}")
    print(f"Val MAE: {val_results['mae']:.2f}")
    print(f"Val RMSE: {val_results['rmse']:.2f}")
    print(f"Val PSNR: {val_results['psnr']:.2f}")
    print(f"Val SSIM: {val_results['ssim']:.4f}")

    # Save best model
    if val_results["mae"] < best_mae:
        best_mae = val_results["mae"]
        best_results = val_results.copy()
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_mae,
            "checkpoints/csrnet_dcn_dual/best_model.pth"
        )


Epoch [1/50]


Validating: 100%|██████████| 316/316 [00:47<00:00,  6.65it/s]


Train Loss: 0.000001
Val MAE: 187.42
Val RMSE: 197.56
Val PSNR: 26.75
Val SSIM: 0.4506

Epoch [2/50]


Validating: 100%|██████████| 316/316 [00:48<00:00,  6.52it/s]


Train Loss: 0.000001
Val MAE: 261.26
Val RMSE: 269.64
Val PSNR: 25.89
Val SSIM: 0.3995

Epoch [3/50]


...

Train Loss: 0.000000
Val MAE: 31.80
Val RMSE: 48.49
Val PSNR: 30.67
Val SSIM: 0.6789

Epoch [49/50]


Validating: 100%|██████████| 316/316 [00:44<00:00,  7.05it/s]


Train Loss: 0.000000
Val MAE: 44.12
Val RMSE: 53.52
Val PSNR: 30.36
Val SSIM: 0.6736

Epoch [50/50]


Validating: 100%|██████████| 316/316 [00:46<00:00,  6.77it/s]

Train Loss: 0.000000
Val MAE: 37.86
Val RMSE: 49.60
Val PSNR: 30.59
Val SSIM: 0.6741


In [ ]:
import pandas as pd

results_df = pd.DataFrame([best_results])
results_df.to_csv("results/csrnet_dcn_dual_metrics.csv", index=False)

print("Saved results.")

Saved results.


In [ ]:
import shutil

shutil.copy(
    "checkpoints/csrnet_dcn_dual/best_model.pth",
    "/kaggle/working/csrnet_dcn_dual_best_model.pth"
)

print("Checkpoint copied to working directory.")

Checkpoint copied to working directory.


In [ ]:
from src.utils.visualization import visualize_predictions

fixed_indices = [165, 173, 33, 78, 93]

visualize_predictions(
    model,
    test_dataset,
    device,
    save_dir="results/qualitative_results/csrnet_dcn_dual",
    indices=fixed_indices
)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].


Saved visualizations to results/qualitative_results/csrnet_dcn_dual
